In [1]:
pdf_path = '2025舞光LED21st(單頁水印可搜尋).pdf'

In [ ]:
import os
from docling.document_converter import DocumentConverter
from tqdm import tqdm

# Initialize Docling Converter
converter = DocumentConverter()
output_md = "catalog_full.md"

def scan_to_markdown():
    print(f"Starting conversion for {pdf_path}...")
    
    # Perform the conversion (Docling handles the full document structure)
    result = converter.convert(pdf_path)
    
    # Iterate through pages and append to a single Markdown file
    with open(output_md, "w", encoding="utf-8") as f:
        for page_no, _ in enumerate(tqdm(result.document.pages, desc="Scanning Pages")):
            # Export one page at a time to Markdown
            page_md = result.document.export_to_markdown(page_numbers=[page_no + 1])
            
            # Label the page for easy identification later
            f.write(f"\n\n\n")
            f.write(page_md)
            f.write(f"\n\n\n---\n")
            print(f'finished page {page_no}')

    print(f"Finished! Total pages processed. File saved to: {output_md}")


scan_to_markdown()

/Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting conversion for 2025舞光LED21st(單頁水印可搜尋).pdf...


In [ ]:
import os
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
# Import PdfFormatOption - this is the missing piece!
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption 

def process_dancelight_docs(input_path, output_md):
    # 1. Setup the Pipeline (The "What" to do)
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_table_structure = True  # Keep this for your LED spec tables
    
    # 2. Setup the Format Option (The "How" to do it)
    # We wrap the pipeline_options here
    format_options = {
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }

    # 3. Initialize the converter with the correct format_options
    converter = DocumentConverter(format_options=format_options)

    # 4. Convert and Export
    result = converter.convert(input_path)
    full_markdown = result.document.export_to_markdown()

    with open(output_md, "w", encoding="utf-8") as f:
        f.write(full_markdown)
    
    print(f"✅ Semantic Markdown saved to: {output_md}")
    return result.document

if __name__ == "__main__":
    output_path = "dancelight_knowledge_base.md"
    
    if os.path.exists(pdf_path):
        doc_object = process_dancelight_docs(pdf_path, output_path)

✅ Semantic Markdown saved to: dancelight_knowledge_base.md


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1. Load the giant Markdown file
with open("catalog_full.md", "r", encoding="utf-8") as f:
    full_text = f.read()

# 2. Split by Header (Hierarchy-aware chunking)
headers_to_split_on = [
    ("#", "Category"),
    ("##", "Series"),
    ("###", "Product_Name"),
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
header_splits = md_splitter.split_text(full_text)

# 3. Sub-splitting (Optional)
# If a single product page is too large, split it further while keeping 10-20% overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
final_chunks = text_splitter.split_documents(header_splits)

# 4. Ingest into Vector Database
vector_db = Chroma.from_documents(
    documents=final_chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./dancelight_vector_db"
)

print(f"Ingested {len(final_chunks)} searchable chunks into ChromaDB.")

ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.

In [ ]:
from langchain_community.query_constructor.base import AttributeInfo
from langchain_community.retrievers import SelfQueryRetriever

# Define the attributes for your LED lights
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The type of light (e.g., Downlight, Track light, Strip light)",
        type="string",
    ),
    AttributeInfo(
        name="ip_rating",
        description="Waterproof rating (e.g., IP20 for indoor, IP65 for bathroom/outdoor)",
        type="string",
    ),
    AttributeInfo(
        name="color_temp",
        description="The light color temperature in Kelvin (e.g., 3000K, 4000K, 6000K)",
        type="integer",
    ),
]

document_content_description = "Specifications and descriptions of DanceLight LED lighting products"

# Now you can create your retriever
retriever = SelfQueryRetriever.from_llm(
    llm=llm, # Your GPT-5 instance
    vectorstore=vector_db, # Your Chroma instance
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True
)

In [ ]:
from langchain_community.retrievers import SelfQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Create the retriever that understands your metadata
retriever = SelfQueryRetriever.from_llm(
    llm,
    vector_db, # Your Chroma database from the previous step
    document_content_description,
    metadata_field_info,
    verbose=True # Shows you the generated filters in the terminal
)

ImportError: cannot import name 'SelfQueryRetriever' from 'langchain_community.retrievers' (/Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/lib/python3.11/site-packages/langchain_community/retrievers/__init__.py)

In [ ]:
from langchain_openai import ChatOpenAI
# These are the modern replacements for v1.2.10
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Setup (assuming your vector_db is already initialized)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model_name="gpt-5.2", temperature=0)

# 2. Define the Prompt (Required in new versions)
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context:
<context>{context}</context>
Question: {input}""")

# 3. Create the Chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# 4. Invoke
response = rag_chain.invoke({"input": "我要找放在浴室的燈?"})
print(response["answer"])